# Amazon Product Intelligence - Exploratory Analysis

Analyze discovered products and validate scoring model

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
%matplotlib inline

## Load Data

In [ ]:
df = pd.read_csv('../data/processed/opportunities.csv')
print(f"Loaded {len(df)} products")
df.head()

## Score Distribution

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

scores = ['total_score', 'demand_score', 'competition_score', 'margin_score', 'simplicity_score', 'price_stability_score']

for idx, score in enumerate(scores):
    ax = axes[idx // 3, idx % 3]
    df[score].hist(bins=20, ax=ax)
    ax.set_title(score.replace('_', ' ').title())
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

## Price vs Sales Rank

In [ ]:
plt.figure(figsize=(12, 6))
plt.scatter(df['current_price'], df['current_sales_rank'], 
            c=df['total_score'], cmap='viridis', alpha=0.6, s=50)
plt.colorbar(label='Total Score')
plt.xlabel('Current Price ($)')
plt.ylabel('Sales Rank')
plt.title('Price vs Sales Rank (colored by opportunity score)')
plt.yscale('log')
plt.show()

## Competition Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df['new_offer_count'], df['total_score'], alpha=0.5)
axes[0].set_xlabel('Number of Sellers')
axes[0].set_ylabel('Total Score')
axes[0].set_title('Sellers vs Opportunity Score')

axes[1].scatter(df['review_count'], df['total_score'], alpha=0.5)
axes[1].set_xlabel('Review Count')
axes[1].set_ylabel('Total Score')
axes[1].set_title('Review Barrier vs Opportunity Score')
axes[1].set_xscale('log')

plt.tight_layout()
plt.show()

## Top Opportunities by Category

In [ ]:
top_categories = df.groupby('category')['total_score'].agg(['count', 'mean']).sort_values('mean', ascending=False).head(10)
print("Top Categories by Average Score:")
print(top_categories)

## Flag Analysis

In [ ]:
flag_counts = {}
for flags_str in df['flags'].dropna():
    for flag in flags_str.split(','):
        flag = flag.strip()
        if flag:
            flag_counts[flag] = flag_counts.get(flag, 0) + 1

flag_df = pd.DataFrame(list(flag_counts.items()), columns=['Flag', 'Count']).sort_values('Count', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(flag_df['Flag'], flag_df['Count'])
plt.xlabel('Count')
plt.title('Product Flags Distribution')
plt.tight_layout()
plt.show()

## Correlation Matrix

In [ ]:
score_cols = ['demand_score', 'competition_score', 'margin_score', 'simplicity_score', 'price_stability_score']
corr = df[score_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0)
plt.title('Score Component Correlations')
plt.tight_layout()
plt.show()